In [ ]:
import os
import re
import json
import time
import shutil
import random
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from urllib.parse import urlparse

import pandas as pd
import requests
from tqdm import tqdm
from dateutil import parser as date_parser

In [ ]:
os.environ["EXA_API_KEY"] = "..."

In [ ]:
EXA_API_URL = "https://api.exa.ai/search"

def parse_dt_safe(dt_str: Optional[str]) -> Optional[datetime]:
    if dt_str is None or pd.isna(dt_str):
        return None
    try:
        dt = date_parser.parse(str(dt_str))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None


def to_iso_z(dt: datetime) -> str:
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def safe_json_loads(x):
    if pd.isna(x):
        return []
    try:
        obj = json.loads(x)
        return obj if isinstance(obj, list) else []
    except Exception:
        return []


def safe_news_len(x) -> int:
    return len(safe_json_loads(x))


def build_search_query(question: str) -> str:
    query = str(question).strip()
    if query.endswith("?"):
        query = query[:-1]
    if query.lower().startswith("will "):
        query = query[5:]
    return query

In [ ]:
# более мягкий поиск для особенных случаев
def build_soft_search_query(question: str) -> str:
    q = str(question).strip()
    q_lower = q.lower()

    # Elon tweets / posts
    if re.search(r"\b(elon|musk)\b", q_lower) and re.search(r"\b(tweet|tweets|post|posts)\b", q_lower):
        return "Elon Musk X Twitter posts activity tweets"

    # General social post count
    if re.search(r"\b(tweet|tweets|post|posts|truth social|instagram|on x)\b", q_lower):
        #пытаемся оставить имя/сущность из вопроса
        entities = extract_entity_phrases_soft(q)
        if entities:
            return " ".join(entities[:3]) + " social media posts tweets activity"
        return build_search_query(q) + " social media posts activity"

    # Eurovision
    if "eurovision" in q_lower:
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:4]) + " Eurovision odds predictions rehearsals"

    # Spotify / charts / App Store
    if re.search(r"\bspotify|app store|itunes|billboard|charts?\b", q_lower):
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:4]) + " Spotify App Store Billboard charts ranking"

    # Box office
    if re.search(r"\bbox office|gross|opening weekend|domestic\b", q_lower):
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:4]) + " box office gross revenue prediction"

    # Awards
    if re.search(r"\boscars?|grammys?|emmys?|golden globes?|academy awards?|best picture|best actor|best actress\b", q_lower):
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:5]) + " awards odds predictions nominations"

    # Crypto / NFT / market questions
    if re.search(r"\bcrypto|cryptocurrency|token|coin|bitcoin|ethereum|doge|etf|binance|coinbase|nft|sotheby|floor price\b", q_lower):
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:5]) + " crypto market news price listing"

    # Release / launch / premiere
    if re.search(r"\brelease|premiere|launch|debut|come out|public release\b", q_lower):
        entities = extract_entity_phrases_soft(q)
        return " ".join(entities[:5]) + " release date launch premiere news"

    return build_search_query(q)

In [ ]:
def extract_entity_phrases_soft(question: str) -> List[str]:
    question = str(question)
    q_lower = question.lower()
    entities = []

    # фразы в кавычках
    quoted = re.findall(r"[\"']([^\"']+)[\"']", question)
    for q in quoted:
        q = q.strip()
        if len(q) >= 2:
            entities.append(q)

    capitalized_phrases = re.findall(
        r"\b(?:[A-Z][a-zA-Z0-9À-ÿ]+(?:\s+|$)){2,}",
        question
    )

    bad_phrases = {
        "Will",
        "Season",
        "Prime Video",
    }

    for phrase in capitalized_phrases:
        phrase = re.sub(r"\s+", " ", phrase.strip())

        if phrase in bad_phrases:
            continue

        if len(phrase) >= 4:
            entities.append(phrase)

    # известные сущности
    known = {
        "elon": ["Elon Musk", "Musk"],
        "musk": ["Elon Musk", "Musk"],
        "eurovision": ["Eurovision"],
        "spotify": ["Spotify"],
        "app store": ["App Store"],
        "billboard": ["Billboard"],
        "openai": ["OpenAI"],
        "sora": ["OpenAI Sora", "Sora"],
        "dune": ["Dune Part Two", "Dune Part 2", "Dune 2"],
        "dune: part 2": ["Dune Part Two", "Dune Part 2", "Dune 2"],
        "cz": ["Changpeng Zhao", "CZ", "Binance"],
        "changpeng": ["Changpeng Zhao", "Binance"],
        "binance": ["Binance"],
        "taylor swift": ["Taylor Swift"],
        "drake": ["Drake"],
        "epstein": ["Epstein"],
        "jimmy kimmel": ["Jimmy Kimmel"],
        "truth social": ["Truth Social"],
        "trump": ["Donald Trump", "Trump"],
    }

    for key, vals in known.items():
        if key in q_lower:
            entities.extend(vals)

    # дедупликация
    seen = set()
    out = []

    for ent in entities:
        ent = str(ent).strip()
        key = ent.lower()

        if ent and key not in seen:
            seen.add(key)
            out.append(ent)

    return out


def get_keywords_soft(question: str) -> List[str]:
    stop_words = {
        "will", "the", "a", "an", "by", "on", "of", "and", "or", "to",
        "receive", "premiere", "release", "released", "first", "episode",
        "season", "limited", "series", "score", "higher", "before", "after",
        "with", "from", "that", "this", "into", "have", "has", "than",
        "over", "under", "less", "more", "least", "most", "between",
        "date", "guide", "video", "prime", "stream", "streaming", "watch",
        "official", "trailer", "teaser", "review", "reviews", "rating",
        "ratings", "year", "name", "named", "say", "says", "make", "made",
        "times", "time", "less", "more", "under", "over",
    }

    raw_words = re.findall(r"[A-Za-z0-9À-ÿ']+", str(question).lower())

    keywords = []

    for word in raw_words:
        word = word.strip("'").lower()

        if len(word) < 3:
            continue

        if word in stop_words:
            continue

        keywords.append(word)

    seen = set()
    out = []

    for word in keywords:
        if word not in seen:
            seen.add(word)
            out.append(word)

    return out

In [ ]:
def soft_relevance_filter(
    news_context: List[Dict[str, Any]],
    question: str,
    min_keyword_matches: int = 1,
    allow_weak_keyword_fallback: bool = True,
) -> List[Dict[str, Any]]:
    entities = [e.lower() for e in extract_entity_phrases_soft(question)]
    keywords = get_keywords_soft(question)

    clean_news = []

    for item in news_context:
        searchable_text = " ".join(
            [
                str(item.get("title") or ""),
                str(item.get("text") or ""),
                str(item.get("url") or ""),
                str(item.get("domain") or ""),
            ]
        ).lower()

        entity_matches = [ent for ent in entities if ent in searchable_text]
        keyword_matches = [kw for kw in keywords if kw in searchable_text]

        #сильная релевантность
        if len(entity_matches) >= 1:
            item["relevance_mode"] = "entity_match"
            item["weak_relevance"] = False
            item["relevance_entity_matches"] = entity_matches
            item["relevance_keyword_matches"] = len(keyword_matches)
            item["relevance_keywords_used"] = keywords
            item["relevance_entities_used"] = entities
            clean_news.append(item)
            continue

        #мягкая релевантность
        if allow_weak_keyword_fallback and len(keyword_matches) >= min_keyword_matches:
            item["relevance_mode"] = "keyword_fallback"
            item["weak_relevance"] = True
            item["relevance_entity_matches"] = []
            item["relevance_keyword_matches"] = len(keyword_matches)
            item["relevance_keywords_used"] = keywords
            item["relevance_entities_used"] = entities
            clean_news.append(item)
            continue

    return clean_news

In [ ]:
def search_exa_news_soft(
    question: str,
    prediction_date: datetime,
    num_results: int = 30,
    max_text_chars: int = 1500,
    sleep_sec: float = 0.25,
) -> List[Dict[str, Any]]:
    api_key = os.environ.get("EXA_API_KEY")

    if not api_key:
        raise RuntimeError("Не найден EXA_API_KEY.")

    if prediction_date.tzinfo is None:
        prediction_date = prediction_date.replace(tzinfo=timezone.utc)

    cutoff_iso = to_iso_z(prediction_date)
    search_query = build_soft_search_query(question)

    payload = {
        "query": search_query,
        "type": "auto",
        "category": "news",
        "numResults": num_results,
        "endPublishedDate": cutoff_iso,
        "contents": {
            "highlights": True,
            "text": {
                "maxCharacters": max_text_chars
            }
        }
    }

    headers = {
        "x-api-key": api_key,
        "Content-Type": "application/json",
    }

    response = requests.post(
        EXA_API_URL,
        headers=headers,
        json=payload,
        timeout=60,
    )

    if response.status_code != 200:
        error_text = response.text[:1000]

        if (
            response.status_code in {401, 402, 403, 429}
            or "NO_MORE_CREDITS" in error_text
            or "exceeded your credits limit" in error_text
            or "credits limit" in error_text
            or "Invalid API key" in error_text
        ):
            raise RuntimeError(f"STOP_API_PROBLEM: Exa API error {response.status_code}: {error_text}")

        raise RuntimeError(f"Exa API error {response.status_code}: {error_text}")

    data = response.json()
    results = data.get("results", [])

    cleaned_results = []

    for item in results:
        cleaned_results.append(
            {
                "title": item.get("title"),
                "url": item.get("url"),
                "publishedDate": item.get("publishedDate"),
                "author": item.get("author"),
                "text": item.get("text"),
                "highlights": item.get("highlights"),
                "score": item.get("score"),
                "provider": "exa_soft",
                "soft_search_query": search_query,
            }
        )

    time.sleep(sleep_sec)

    return cleaned_results

In [ ]:
def filter_news_before_prediction_date(
    news_context: List[Dict[str, Any]],
    prediction_date: datetime,
) -> List[Dict[str, Any]]:
    if prediction_date.tzinfo is None:
        prediction_date = prediction_date.replace(tzinfo=timezone.utc)

    prediction_date = prediction_date.astimezone(timezone.utc)

    clean_news = []

    for item in news_context:
        pub_dt = parse_dt_safe(item.get("publishedDate"))

        if pub_dt is None:
            continue

        if pub_dt >= prediction_date:
            continue

        clean_news.append(item)

    return clean_news


def check_news_dates(
    news_context: List[Dict[str, Any]],
    prediction_date: datetime,
) -> bool:
    if not news_context:
        return False

    if prediction_date.tzinfo is None:
        prediction_date = prediction_date.replace(tzinfo=timezone.utc)

    prediction_date = prediction_date.astimezone(timezone.utc)

    for item in news_context:
        pub_dt = parse_dt_safe(item.get("publishedDate"))

        if pub_dt is None:
            return False

        if pub_dt >= prediction_date:
            return False

    return True

In [ ]:
def make_run_snapshot(
    output_csv_path: Path,
    run_dir: Path = Path("runs"),
    run_name: str = "soft_retry",
    note: str = "",
) -> Path:
    run_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    snapshot_path = run_dir / f"{run_name}_{timestamp}.csv"

    shutil.copy(output_csv_path, snapshot_path)

    df = pd.read_csv(output_csv_path, low_memory=False)

    log_row = {
        "timestamp": timestamp,
        "run_name": run_name,
        "snapshot_path": str(snapshot_path),
        "note": note,
        "rows": len(df),
        "unique_row_index": df["row_index"].nunique() if "row_index" in df.columns else None,
        "rows_with_news": int((pd.to_numeric(df["filtered_news_count"], errors="coerce").fillna(0).astype(int) > 0).sum())
            if "filtered_news_count" in df.columns else None,
        "rows_without_news": int((pd.to_numeric(df["filtered_news_count"], errors="coerce").fillna(0).astype(int) == 0).sum())
            if "filtered_news_count" in df.columns else None,
        "errors": int(df["news_error"].notna().sum()) if "news_error" in df.columns else None,
    }

    log_path = run_dir / "run_log.csv"
    log_df = pd.DataFrame([log_row])

    if log_path.exists():
        old_log = pd.read_csv(log_path)
        log_df = pd.concat([old_log, log_df], ignore_index=True)

    log_df.to_csv(log_path, index=False, encoding="utf-8")

    print("Snapshot saved to:", snapshot_path)
    print("Run log updated:", log_path)

    return snapshot_path

In [ ]:
# более мягкими штуками зполняем пустые строки
def retry_empty_rows_with_soft_exa(
    csv_path: Path,
    backup_path: Optional[Path] = None,
    max_rows: Optional[int] = None,
    raw_num_results: int = 30,
    final_num_results: int = 8,
    max_text_chars: int = 1500,
    min_keyword_matches: int = 1,
    save_every: int = 10,
    only_without_errors: bool = True,
    target_mode: str = "empty",
    make_snapshot_after_run: bool = True,
    run_note: str = "",
) -> None:
    if backup_path is None:
        backup_path = csv_path.with_name(csv_path.stem + "_BACKUP_before_soft_retry.csv")

    if not backup_path.exists():
        shutil.copy(csv_path, backup_path)
        print("Backup saved to:", backup_path)
    else:
        print("Backup already exists:", backup_path)

    df = pd.read_csv(csv_path, low_memory=False)

    # нужные колонки
    for col in [
        "raw_news_count",
        "date_filtered_news_count",
        "relevance_filtered_news_count",
        "filtered_news_count",
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    if "news_provider" not in df.columns:
        df["news_provider"] = None

    if "fallback_used" not in df.columns:
        df["fallback_used"] = False

    if "fallback_error" not in df.columns:
        df["fallback_error"] = None

    if "soft_retry_attempted" not in df.columns:
        df["soft_retry_attempted"] = False

    if "soft_retry_success" not in df.columns:
        df["soft_retry_success"] = False

    if "soft_retry_error" not in df.columns:
        df["soft_retry_error"] = None

    df["news_len_check"] = df["news_context"].apply(safe_news_len)

    empty_base = (
        (df["filtered_news_count"] == 0)
        | (df["news_len_check"] == 0)
    )

    if target_mode == "empty":
        target_mask = empty_base

    elif target_mode == "relevance_failed":
        target_mask = (
            empty_base
            & (df["date_filtered_news_count"] > 0)
            & (df["relevance_filtered_news_count"] == 0)
        )

    elif target_mode == "api_errors_empty":
        target_mask = (
            empty_base
            & df["news_error"].notna()
        )

    else:
        raise ValueError("target_mode must be one of: empty, relevance_failed, api_errors_empty")

    if only_without_errors:
        target_mask = target_mask & df["news_error"].isna()

    target_mask = target_mask & (~df["soft_retry_success"].astype(str).str.lower().eq("true"))

    retry_indices = df.index[target_mask].tolist()

    if max_rows is not None:
        retry_indices = retry_indices[:max_rows]

    print("Всего строк в файле:", len(df))
    print("target_mode:", target_mode)
    print("Строк для soft retry:", len(retry_indices))

    attempted = 0
    improved = 0
    still_empty = 0
    errors = 0
    stopped_by_api_problem = False

    for idx in tqdm(retry_indices, desc="Soft Exa retry"):
        row = df.loc[idx]

        question = row.get("question")
        prediction_dt = parse_dt_safe(row.get("prediction_date"))

        if pd.isna(question) or prediction_dt is None:
            df.loc[idx, "soft_retry_attempted"] = True
            df.loc[idx, "soft_retry_success"] = False
            df.loc[idx, "soft_retry_error"] = "missing question or prediction_date"
            errors += 1
            attempted += 1
            continue

        try:
            raw_news = search_exa_news_soft(
                question=question,
                prediction_date=prediction_dt,
                num_results=raw_num_results,
                max_text_chars=max_text_chars,
            )

            date_clean_news = filter_news_before_prediction_date(
                news_context=raw_news,
                prediction_date=prediction_dt,
            )

            relevant_news = soft_relevance_filter(
                news_context=date_clean_news,
                question=question,
                min_keyword_matches=min_keyword_matches,
                allow_weak_keyword_fallback=True,
            )

            final_news = relevant_news[:final_num_results]

            df.loc[idx, "raw_news_count"] = len(raw_news)
            df.loc[idx, "date_filtered_news_count"] = len(date_clean_news)
            df.loc[idx, "relevance_filtered_news_count"] = len(relevant_news)
            df.loc[idx, "filtered_news_count"] = len(final_news)

            df.loc[idx, "news_context"] = json.dumps(final_news, ensure_ascii=False)
            df.loc[idx, "news_dates_ok"] = check_news_dates(final_news, prediction_dt)

            df.loc[idx, "soft_retry_attempted"] = True
            df.loc[idx, "soft_retry_error"] = None

            df.loc[idx, "fallback_used"] = True
            df.loc[idx, "news_provider"] = "exa_soft"

            if len(final_news) > 0:
                df.loc[idx, "news_error"] = None
                df.loc[idx, "fallback_error"] = None
                df.loc[idx, "soft_retry_success"] = True
                improved += 1
            else:
                df.loc[idx, "fallback_error"] = "soft Exa retry returned no relevant news"
                df.loc[idx, "soft_retry_success"] = False
                still_empty += 1

            attempted += 1

        except Exception as e:
            error_msg = str(e)

            if (
                "STOP_API_PROBLEM" in error_msg
                or "NO_MORE_CREDITS" in error_msg
                or "exceeded your credits limit" in error_msg
                or "credits limit" in error_msg
                or "Invalid API key" in error_msg
                or "401" in error_msg
                or "402" in error_msg
                or "403" in error_msg
                or "429" in error_msg
            ):
                print("\nAPI problem. Останавливаю запуск, чтобы не записывать мусор.")
                print("Ошибка:", error_msg[:500])
                stopped_by_api_problem = True
                break

            df.loc[idx, "soft_retry_attempted"] = True
            df.loc[idx, "soft_retry_success"] = False
            df.loc[idx, "soft_retry_error"] = error_msg
            errors += 1
            attempted += 1

        if attempted > 0 and attempted % save_every == 0:
            df.drop(columns=["news_len_check"], errors="ignore").to_csv(
                csv_path,
                index=False,
                encoding="utf-8",
            )
            print(
                f"Progress saved. attempted={attempted}, "
                f"improved={improved}, still_empty={still_empty}, errors={errors}"
            )

    # финальное сохранение
    df.drop(columns=["news_len_check"], errors="ignore").to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
    )

    print("Done.")
    print("Attempted:", attempted)
    print("Improved rows:", improved)
    print("Still empty:", still_empty)
    print("Errors:", errors)
    print("Stopped by API problem:", stopped_by_api_problem)
    print("Updated file:", csv_path)

    if make_snapshot_after_run:
        make_run_snapshot(
            output_csv_path=csv_path,
            run_name="soft_exa_retry",
            note=run_note or f"target_mode={target_mode}, raw_num_results={raw_num_results}, min_keyword_matches={min_keyword_matches}",
        )

In [ ]:
retry_empty_rows_with_soft_exa(
    csv_path=Path("merged_culture_dataset_with_news_full.csv"),
    max_rows=1000,
    raw_num_results=30,
    final_num_results=8,
    max_text_chars=1500,
    min_keyword_matches=1,
    save_every=10,
    only_without_errors=True,
    target_mode="relevance_failed",
    make_snapshot_after_run=True,
    run_note="soft retry 1000 relevance_failed rows with first key",
)

Backup saved to: merged_culture_dataset_with_news_full_BACKUP_before_soft_retry.csv
Всего строк в файле: 1996
target_mode: relevance_failed
Строк для soft retry: 195


Soft Exa retry:   5%|▌         | 10/195 [00:20<06:15,  2.03s/it]

Progress saved. attempted=10, improved=10, still_empty=0, errors=0


Soft Exa retry:  10%|█         | 20/195 [01:14<12:53,  4.42s/it]

Progress saved. attempted=20, improved=14, still_empty=0, errors=6


...

Progress saved. attempted=180, improved=171, still_empty=2, errors=7


Soft Exa retry:  97%|█████████▋| 190/195 [06:57<00:10,  2.16s/it]

Progress saved. attempted=190, improved=181, still_empty=2, errors=7


Soft Exa retry: 100%|██████████| 195/195 [07:06<00:00,  2.19s/it]


Done.
Attempted: 195
Improved rows: 186
Still empty: 2
Errors: 7
Stopped by API problem: False
Updated file: merged_culture_dataset_with_news_full.csv
Snapshot saved to: runs/soft_exa_retry_20260521_200821.csv
Run log updated: runs/run_log.csv
